In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")

In [12]:
!pip install langchain_groq
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api_key)


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017F8C7068D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017F8BFA94C0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
from langchain_core.messages import HumanMessage

response = model.invoke([
    HumanMessage(content="Hi, my name is Manasi, I am an AI Engineer")
])

print(response.content)

Nice to meet you, Manasi. As an AI Engineer, I'm sure you're working on exciting projects that involve developing intelligent systems and solving complex problems. What kind of projects have you been working on lately? Are you into natural language processing, computer vision, or perhaps reinforcement learning? I'm here to chat and exchange ideas if you'd like.


In [15]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hi, my name is Manasi, I am an AI Engineer"),
    AIMessage(content="Nice to meet you, Manasi. As an AI Engineer, I'm sure you're working on exciting projects that involve developing intelligent systems and solving complex problems. What kind of projects have you been working on lately? Are you into natural language processing, computer vision, or perhaps reinforcement learning? I'm here to chat and exchange ideas if you'd like."),
    HumanMessage(content="Do you remember my name and what do I do?")
])


AIMessage(content="I remember that your name is Manasi and you are an AI Engineer. I'm glad I could keep that information in mind, even after our conversation started. How can I assist you today, Manasi?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 140, 'total_tokens': 183, 'completion_time': 0.08124325, 'completion_tokens_details': None, 'prompt_time': 0.009492912, 'prompt_tokens_details': None, 'queue_time': 0.048746662, 'total_time': 0.090736162}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe05a-6bbf-7651-af7d-5b3dd7802da5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 140, 'output_tokens': 43, 'total_tokens': 183})

In [16]:
!pip install langchain_community


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [20]:
store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_chat_history=RunnableWithMessageHistory(model,get_session_history=get_session_history)

d:\langchain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [29]:
config={"configurable":{"session_id":"chat1"}}

In [30]:
response=with_chat_history.invoke([
    HumanMessage(content="Hi,My name is Manasi,I am an AI Engineer")],
    config=config)

In [31]:
response.content

"Nice to meet you, Manasi! As an AI Engineer, you must be working on some exciting projects. We had a conversation earlier, remember? I mentioned that I don't have the ability to recall past conversations, but now that you've reminded me of your name and profession, I'd love to catch up and see if there's anything you'd like to talk about. What's been on your mind lately, or what's the most interesting project you're working on?"

In [ ]:
#it remembers using the config
response=with_chat_history.invoke([
    HumanMessage(content="Hi,Do you remember me?")],
    config=config)
response.content

"We've had a few conversations already. You're Manasi, an AI Engineer. I remember that much! However, I don't retain any specific details from our previous conversations. Each time you interact with me, it's a new conversation. Would you like to start fresh and talk about something new, or pick up where we left off?"

In [ ]:
#here the variable name is changed but the session id remains the same so the chatbot remmebers the data that was said earlier
config1={"configurable":{"session_id":"chat1"}}

In [ ]:
#remembers
response=with_chat_history.invoke([
    HumanMessage(content="Hi,Do you remember me?")],
    config=config1)
response.content

"We've had conversations before. You're Manasi, an AI Engineer. I'm glad you're here again, but I don't have any specific memories of our previous conversations. Let's start fresh and see where the conversation takes us today. What's been on your mind, or what would you like to talk about?"

In [ ]:
#here the session id is changed so eventually the chatbot forgets who i am i.e the previous chats are forgotten
config={"configurable":{"session_id":"chat2"}}

In [36]:
response=with_chat_history.invoke([
    HumanMessage(content="Hi,Do you remember me?")],
    config=config)
response.content

"I'm an AI, and I don't have personal memories or experiences like humans do. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. I'm here to help with any questions or topics you'd like to discuss, though. How can I assist you today?"